# 🚀 Notebook 2: Database Optimization

Before adding caching or replicas, optimize your existing database. Proper indexing solves most read scaling problems.

## Learning Objectives

By the end of this notebook, you'll understand:
- How indexes work (B-tree, Hash)
- Creating effective indexes
- Composite indexes and column order
- Reading EXPLAIN output

---

🔍 **Open Adminer** at http://localhost:8080 to run queries and see execution plans!

In [ ]:
import psycopg2
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

def run_query(query: str):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(query)
    try:
        results = cursor.fetchall()
    except:
        results = []
    conn.commit()
    conn.close()
    return results

def measure_query(query: str) -> tuple:
    conn = get_connection()
    cursor = conn.cursor()
    start = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = (time.time() - start) * 1000
    conn.close()
    return elapsed, len(results)

def explain_query(query: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(f"EXPLAIN ANALYZE {query}")
    plan = [row[0] for row in cursor.fetchall()]
    conn.close()
    return plan

print("✅ Connected to PostgreSQL")

## 📚 How Indexes Work

An index is like a book's index - instead of reading every page, you look up where to find what you need.

In [ ]:
print("📚 Index Types")
print("=" * 60)
print("""
B-TREE INDEX (Default)
─────────────────────────────────────────────────────────────
• Best for: Range queries, equality, sorting
• Supports: <, <=, =, >=, >, BETWEEN, LIKE 'prefix%'
• Structure: Balanced tree with O(log n) lookups

              [50]
             /    \\
         [25]      [75]
        /    \\    /    \\
     [10]  [30] [60]  [90]

Example: Finding email = 'user50@example.com'
         Only checks ~3-4 nodes instead of 100 rows!

─────────────────────────────────────────────────────────────

HASH INDEX
─────────────────────────────────────────────────────────────
• Best for: Exact equality only (=)
• Does NOT support: Range queries, sorting
• Structure: Hash table with O(1) lookups

hash('user50@example.com') → bucket[42] → row pointer

─────────────────────────────────────────────────────────────
""")

print("💡 Use B-tree (default) unless you ONLY do exact matches!")

## 🔬 Before and After: Index Impact

In [ ]:
run_query("DROP INDEX IF EXISTS idx_users_email")

print("🔬 BEFORE INDEX: Find user by email")
print("=" * 60)

query = "SELECT * FROM users WHERE email = 'user50@example.com'"

elapsed, count = measure_query(query)
print(f"\nTime: {elapsed:.2f}ms | Rows: {count}")
print("\nExecution Plan:")
for line in explain_query(query):
    print(f"  {line}")

In [ ]:
print("🔨 Creating index on users.email...")
run_query("CREATE INDEX idx_users_email ON users(email)")
print("✅ Index created!\n")

print("🔬 AFTER INDEX: Find user by email")
print("=" * 60)

elapsed, count = measure_query(query)
print(f"\nTime: {elapsed:.2f}ms | Rows: {count}")
print("\nExecution Plan:")
for line in explain_query(query):
    print(f"  {line}")

print("\n💡 Notice 'Index Scan' instead of 'Seq Scan'!")

## 🎯 Composite Indexes

When queries filter on multiple columns, composite indexes help.

In [ ]:
print("🎯 Composite Index: Products by category and price")
print("=" * 60)

query = """
SELECT * FROM products 
WHERE category = 'Electronics' AND price < 100
ORDER BY price
"""

run_query("DROP INDEX IF EXISTS idx_products_category_price")

print("\nBEFORE composite index:")
elapsed, count = measure_query(query)
print(f"Time: {elapsed:.2f}ms | Rows: {count}")
for line in explain_query(query)[:3]:
    print(f"  {line}")

In [ ]:
print("\n🔨 Creating composite index (category, price)...")
run_query("CREATE INDEX idx_products_category_price ON products(category, price)")

print("\nAFTER composite index:")
elapsed, count = measure_query(query)
print(f"Time: {elapsed:.2f}ms | Rows: {count}")
for line in explain_query(query)[:3]:
    print(f"  {line}")

## ⚠️ Column Order Matters!

In [ ]:
print("⚠️ Composite Index Column Order")
print("=" * 60)
print("""
Index on (category, price) supports:
─────────────────────────────────────────────────────────────
✅ WHERE category = 'X'                    (leftmost column)
✅ WHERE category = 'X' AND price < 100    (both columns)
❌ WHERE price < 100                       (skips leftmost!)

Think of it like a phone book:
─────────────────────────────────────────────────────────────
• Sorted by (LastName, FirstName)
• Easy to find all "Smith"s
• Easy to find "Smith, John"
• Hard to find all "John"s (scattered throughout!)
""")

print("\n🔬 Query using ONLY price (skips category):")
query_price_only = "SELECT * FROM products WHERE price < 50"
for line in explain_query(query_price_only)[:2]:
    print(f"  {line}")
print("\n💡 Falls back to Seq Scan because price isn't leftmost!")

## 📊 Index Recommendations

In [ ]:
print("📊 When to Create Indexes")
print("=" * 60)
print("""
✅ DO INDEX:
─────────────────────────────────────────────────────────────
• Primary keys (automatic)
• Foreign keys used in JOINs
• Columns in WHERE clauses
• Columns in ORDER BY
• Columns with high cardinality (many unique values)

❌ DON'T INDEX:
─────────────────────────────────────────────────────────────
• Tiny tables (< 1000 rows)
• Columns with low cardinality (e.g., boolean, status)
• Columns rarely used in queries
• Tables with heavy writes and few reads

⚖️ TRADE-OFFS:
─────────────────────────────────────────────────────────────
• Indexes speed up reads but slow down writes
• Each index adds storage overhead
• Too many indexes → slower INSERT/UPDATE/DELETE
• For read-heavy apps, index liberally!
""")

In [ ]:
print("📋 Creating Common Indexes for Our Schema")
print("=" * 60)

indexes = [
    ("idx_posts_user_id", "posts(user_id)", "Find posts by author"),
    ("idx_posts_created_at", "posts(created_at DESC)", "Recent posts"),
    ("idx_comments_post_id", "comments(post_id)", "Comments on a post"),
    ("idx_reviews_product_id", "reviews(product_id)", "Reviews for product"),
    ("idx_short_urls_code", "short_urls(short_code)", "URL lookup"),
]

for idx_name, idx_def, description in indexes:
    run_query(f"DROP INDEX IF EXISTS {idx_name}")
    run_query(f"CREATE INDEX {idx_name} ON {idx_def}")
    print(f"✅ {idx_name}: {description}")

print("\n💡 These indexes will help all our subsequent queries!")

## 🔍 Query Optimization Tips

In [ ]:
print("🔍 Query Optimization Tips")
print("=" * 60)
print("""
1. SELECT ONLY WHAT YOU NEED
─────────────────────────────────────────────────────────────
❌ SELECT * FROM users WHERE id = 1
✅ SELECT username, email FROM users WHERE id = 1

2. USE LIMIT FOR PAGINATION
─────────────────────────────────────────────────────────────
❌ SELECT * FROM posts ORDER BY created_at DESC
✅ SELECT * FROM posts ORDER BY created_at DESC LIMIT 20

3. AVOID FUNCTIONS ON INDEXED COLUMNS
─────────────────────────────────────────────────────────────
❌ WHERE LOWER(email) = 'user@example.com'  -- Can't use index
✅ WHERE email = 'user@example.com'          -- Uses index

4. USE EXISTS INSTEAD OF COUNT FOR EXISTENCE CHECKS
─────────────────────────────────────────────────────────────
❌ SELECT COUNT(*) FROM likes WHERE post_id = 1  -- Scans all
✅ SELECT EXISTS(SELECT 1 FROM likes WHERE post_id = 1)  -- Stops early

5. BATCH QUERIES WHEN POSSIBLE
─────────────────────────────────────────────────────────────
❌ Loop: SELECT * FROM users WHERE id = 1, 2, 3...
✅ SELECT * FROM users WHERE id IN (1, 2, 3, ...)
""")

## 🧪 Quick Quiz

1. **What's the difference between B-tree and Hash indexes?**

2. **For index (A, B, C), which queries can use it?**

3. **Why not index every column?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. B-tree vs Hash:")
print("   B-tree: Range queries, sorting, equality")
print("   Hash: Only equality (=), faster for exact match")
print("   Default to B-tree unless you know otherwise")
print()
print("2. Index (A, B, C) supports:")
print("   ✅ WHERE A = x")
print("   ✅ WHERE A = x AND B = y")
print("   ✅ WHERE A = x AND B = y AND C = z")
print("   ❌ WHERE B = y (skips A)")
print("   ❌ WHERE C = z (skips A, B)")
print()
print("3. Why not index everything:")
print("   - Slows down INSERT/UPDATE/DELETE")
print("   - Uses disk space")
print("   - Index maintenance overhead")
print("   - For read-heavy apps, still index liberally!")

## 📚 Summary

### Key Takeaways

1. **Indexes turn O(n) into O(log n)** - massive speedup
2. **Use EXPLAIN** - see how queries actually execute
3. **Column order matters** - leftmost columns used first
4. **Index for your queries** - not generic "best practices"
5. **B-tree is usually right** - handles most use cases

### Next Up

In **Notebook 3**, we'll learn about denormalization:
- Trading storage for speed
- Materialized views
- Pre-computed aggregations